In [2]:
import sys
import os.path as osp

PROJECT_DIR = '../../'
PROJECT_DIR = osp.abspath(PROJECT_DIR)
print(PROJECT_DIR in sys.path)
if PROJECT_DIR not in sys.path:
    print(f'Adding project directory to the sys.path: {PROJECT_DIR!r}')
    sys.path.insert(1, PROJECT_DIR)

True


We will build our initial two-stage recommender system based on the two steps:

- **candidate generation** with item-based collaborative filtering algrotithm based on the Alibaba's Swing algorithm [[1]](https://www.alibabacloud.com/help/en/airec/airec/user-guide/industry-algorithm-models) [[2]](https://eugeneyan.com/writing/real-time-recommendations/#how-to-design-and-implement-an-mvp) [[3]](https://arxiv.org/pdf/2010.05525). The candidate generation itself has two pars, offline and online stages. Let's assume that we make a recommendation for the user $U_1$.
  - First, all of the item-item pairs are scored according to the formulas:
    - User weight: $w_u = \frac{1}{(n_u + 5)^0.35}$, where $n_u$ is the number of the user's interactions (number of movies the respective user has rated). The idea behind it is that the users who have rated less items, and therefore are only at the start of their ratings, are more likely to have these first movies as their most desired and to be more selective; while the users with a lot of ratings are less meaningful for the similar item selection as they have rated a lot of the item varieties.
    - User pair weight: $w_{pair_{i,j}} = w_i*w_j$
    - Item-item score: $s_{k,l} = \sum_{pairs} \frac{w_{pair_{i,j}}}{n_{both_{i,j}}}$, where $(u_i,u_j)$ are users that have both rated items $k$ and $l$, and $n_{both_{i,j}}$ is the number of items both users have ranked. Essentially, we measure how influental for the item similarity are all of the user pairs that have rated both these elements, and the pairs where each of the users has rated less movies in total have higher sum members. Notice how we can increment these scores with each new user, which gives an advantage to this scoring method.
  - Then, when we receive an online query in the form of the user $U_1$ for all items we calculate the sum of item-item scores with each item that the user $U_1$ has rated. The top-$N$ items with the lowest scores are our candidates to be passed to the ranking step. Here, we initially take the $N=50$ as the cutoff as it allows to both capture the diverse selection of the candidates, and filter only the relevant items for the ranking step. The ranking step in most of the two-stage systems is a computation-heavy step, so such a selection serves a puprose to make the recommendation more lightweight and quick. Additionally, the top-$M$ items closest to the each movie rated 4 or 5 by the user $U_1$ will also be added to the candidate list, with the $M$ selected as 5 here. Also, notice how the current step of generating candidates is very quick due to the item-item scores being pre-computed.
- Then, on the **ranking step**, we also have an offline and an online parts:
  - On the offline step, we score all of the user-user pairs according to the cosine similarity score.
  - On the online step, we assign score to all of the candidates taking into account the computed earlier similarity scores to all of the movies watched be the user $U_1$, ratings of the movies already watched by the user $U_1$, genres of the both candidate items and the already watched movies, and also whether the candidate items were rated by the most similar users. Essentially, we use almost all possible features, available to us in the dataset.

After developing the initial two-stage model, we'll evaluate it in the evaluation framework, and then try to do several additional model development interations in order to further improve the performance.

With our plan outline, let's realize both of the steps and create our initial two-step models.

In [3]:
from abc import ABC, abstractmethod

In [8]:
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm
import json
import heapq

In [6]:
df_ratings = pd.read_csv('../../data/ml-1m/ratings.dat',
                         delimiter='::',
                         header=None,
                         names=['UserID','MovieID','Rating','Timestamp'],
                         engine ='python')

In [9]:
df_users = pd.read_csv('../../data/ml-1m/users.dat',
                         delimiter='::',
                         header=None,
                         names=['UserID','Gender','Age','Occupation','Zip-code'],
                         engine ='python')

In [12]:
df_movies = pd.read_csv('../../data/ml-1m/movies.dat',
                         delimiter='::',
                         header=None,
                         names=['UserID','Gender','Age','Occupation','Zip-code'],
                         engine ='python',
                         encoding='latin-1')

In [16]:
sparse_ratings = scipy.sparse.csr_matrix((df_ratings['Rating'],
                                          (df_ratings['UserID'] - 1, df_ratings['MovieID'] - 1)))
sparse_ratings

<6040x3952 sparse matrix of type '<class 'numpy.int64'>'
	with 1000209 stored elements in Compressed Sparse Row format>

In [39]:
print(sparse_ratings[0].multiply(sparse_ratings[1]))

  (0, 1192)	25
  (0, 1206)	16
  (0, 1245)	20
  (0, 1961)	20
  (0, 2027)	20
  (0, 2320)	9
  (0, 3104)	20


In [63]:
i2i = {}
for i in tqdm(range(sparse_ratings.shape[0])):
    wi = np.power(sparse_ratings[i].shape[0] + 5, -0.35)
    for j in range(i + 1, sparse_ratings.shape[0]):
        intersection = sparse_ratings[i].multiply(sparse_ratings[j]).tocoo().col
        wj = wi * np.power(sparse_ratings[j].shape[0] + 5, -0.35)
        for i_p, product_id in enumerate(intersection):
            for product_id_2 in intersection[i_p+1:]:
                i2i[str((product_id,product_id_2))] = i2i.get(str((product_id,product_id_2)), 0.0) + wj / (1 + len(intersection))

  0%|          | 0/6040 [00:00<?, ?it/s]

In [64]:
len(i2i)

4922072

In [65]:
with open('i2i_scores.json', 'w') as f:
    json.dump(i2i, f)

But let's remember, that these are the static scores for the latest point in time. In order to properly evaluate the model, we need to calculate similarity as it was at the various points in time, then maske the predictions based on it. Therefore, we need to develop a pipeline for the model to perform this progressive item similarity updates.

Let's therefore build a full initial two-stage model:

In [7]:
from src.models.abstract_rs_model import AbstractRSModel

In [27]:
class TwoStageSwingBasedModel(AbstractRSModel):
    def __init__(self, previous_rows_before_timestamp = 0, i2i_file: str = ''):
        self.pre_fit = False
        self.previous_rows_before_timestamp = previous_rows_before_timestamp
        self.items_rated_by_user = {}
        self.users_rated_the_item = {}
        self.user_item_ratings = {}
        if len(i2i_file) == 0:
            # No pre-calculated initial weights are provided:
            self.i2i = {}
    
    def fit(self, train_data, pre_fit: bool = False):
        if self.pre_fit:
            # The train data was already pre-fit
            pass
        else:
            pass
        self.pre_fit = pre_fit

    def predict(self, data_at_test_timestamp, test_user, test_timestamp):
        # data_at_test_timestamp.shape[0] here is the number of rows
        # before the data for the timestamp in question ends
        if data_at_test_timestamp.shape[0] > self.previous_rows_before_timestamp:
            self._update_i2i_scores(data_at_test_timestamp[self.previous_rows_before_timestamp:])
            self.previous_rows_before_timestamp = data_at_test_timestamp.shape[0]
        # Candidate generation step
        candidate_items = []
        aggregate_scores = {}
        # print(f'--{self.i2i.keys()}')
        if test_user in self.items_rated_by_user:
            # print(f'--{self.items_rated_by_user[test_user]}')
            for item_already_rated in self.items_rated_by_user[test_user]:
                if item_already_rated in self.i2i.keys():
                    for item_compared in self.i2i[item_already_rated]:
                        aggregate_scores[item_compared] = aggregate_scores.get(item_compared, 0.0) + self.i2i[
                            item_already_rated][item_compared]
                    if self.user_item_ratings[test_user][item_already_rated] >= 4:
                        candidate_items += heapq.nsmallest(5,
                                                               self.i2i[item_already_rated],
                                                               key=self.i2i[item_already_rated].get)
        candidate_items += heapq.nsmallest(50,
                                               aggregate_scores,
                                               key=aggregate_scores.get)
        candidate_items = list(np.unique(candidate_items))
        # Ranking step
        aggregate_ratings = {}
        if test_user in self.items_rated_by_user:
            # print(f'--{self.items_rated_by_user[test_user]}')
            for item_already_rated in self.items_rated_by_user[test_user]:
                if item_already_rated in self.i2i.keys():
                    for candidate_item in candidate_items:
                        if candidate_item in self.i2i[item_already_rated].keys():
                            # Can also potentially include user-user similarity, and here
                            # it was decided to perform the weighted averaging based on
                            # the ratings to increase the rating precision
                            aggregate_ratings[candidate_item] = aggregate_ratings.get(candidate_item, 0.0) + self.i2i[
                                item_already_rated][candidate_item]*self.user_item_ratings[test_user][item_already_rated]
        ranked_candidates = np.array(sorted(aggregate_ratings, key=aggregate_ratings.get))
        ranked_candidates_scores = np.array(sorted(aggregate_ratings.values()))
        if len(ranked_candidates_scores) > 0:
            if len(ranked_candidates_scores) > 1:
                # print(ranked_candidates_scores.max() - ranked_candidates_scores.min(), (ranked_candidates_scores - ranked_candidates_scores.min())/(
                #     ranked_candidates_scores.max() - ranked_candidates_scores.min()), ranked_candidates_scores.max(), ranked_candidates_scores.min(), ranked_candidates_scores)
                ranked_candidates_scores = -(ranked_candidates_scores - ranked_candidates_scores.min())/(
                    ranked_candidates_scores.max() - ranked_candidates_scores.min()) + 5
            else:
                ranked_candidates_scores = ranked_candidates_scores/ranked_candidates_scores + 4
        return ranked_candidates, ranked_candidates_scores

    def fit_predict(self, data, test_user, test_timestamp):
        self.fit(data)
        return self.predict(data, test_user, test_timestamp)

    def _update_i2i_scores(self, new_data_at_test_timestamp):
        # for i_p, point in tqdm(new_data_at_test_timestamp.iterrows(),
        #                             total=new_data_at_test_timestamp.shape[0]):
        for i_p, point in new_data_at_test_timestamp.iterrows():
            # if point['UserID'] in [6035, 6036] and point['MovieID'] in [25,32]:
            #     print(point)
            #     print(self.items_rated_by_user[point['UserID']])
            # for i_p, point in new_data_at_test_timestamp.iterrows():
            if point['UserID'] not in self.items_rated_by_user:
                self.items_rated_by_user[point['UserID']] = []
                self.user_item_ratings[point['UserID']] = {}
            for item_already_rated in self.items_rated_by_user[point['UserID']]:
                for user_id in self.items_rated_by_user.keys():
                    # if point['UserID'] == 6036 and point['MovieID'] == 32: # and item_already_rated == 26:
                    #     # if point['UserID'] in [6035, 6036] and point['MovieID'] in [25,32] and user_id in [6035, 6036]:
                    #     print(item_already_rated, user_id, (point['MovieID'] in self.items_rated_by_user[user_id]), (
                    #         item_already_rated in self.items_rated_by_user[user_id]))
                    if (point['MovieID'] in self.items_rated_by_user[user_id]) and (item_already_rated in self.items_rated_by_user[user_id]):
                        # item_id_1 = min(point['MovieID'], item_already_rated)
                        # item_id_2 = max(point['MovieID'], item_already_rated)
                        intersection = set(self.items_rated_by_user[point['UserID']]) & set(self.items_rated_by_user[user_id])
                        wj = np.power(len(self.items_rated_by_user[point['UserID']]) + 6, -0.35) * np.power(len(self.items_rated_by_user[user_id]) + 5, -0.35)
                        # Notice how here we don't update the users' ratings on each step. This allows to
                        # assign higher weights to the earlier ratings by each user
                        if point['MovieID'] not in self.i2i.keys():
                            self.i2i[point['MovieID']] = {}
                        if item_already_rated not in self.i2i.keys():
                            self.i2i[item_already_rated] = {}
                        self.i2i[point['MovieID']][item_already_rated] = self.i2i[point['MovieID']].get(item_already_rated, 0.0) + wj / (1 + len(intersection))
                        self.i2i[item_already_rated][point['MovieID']] = self.i2i[item_already_rated].get(point['MovieID'], 0.0) + wj / (1 + len(intersection))
            self.items_rated_by_user[point['UserID']].append(point['MovieID'])
            self.user_item_ratings[point['UserID']][point['MovieID']] = point['Rating']

In [292]:
items_pred, ratings_pred = TwoStageSwingBasedModel().fit_predict(df_ratings[
                    df_ratings['Timestamp'] < 956716294], 6035, 956716294) # 1028

  0%|          | 0/1137 [00:00<?, ?it/s]

In [293]:
print(list(zip(items_pred[:20], ratings_pred[:20])))

[(899, 5.0), (2300, 4.999840101055492), (1294, 4.999136175188606), (750, 4.9989519609976085), (1282, 4.998770196318617), (1269, 4.998536030771286), (900, 4.998320589096095), (903, 4.998320589096095), (916, 4.998320589096095), (930, 4.998320589096095), (945, 4.998320589096095), (951, 4.998320589096095), (1188, 4.998320589096095), (1197, 4.998320589096095), (1211, 4.998320589096095), (2208, 4.998320589096095), (2565, 4.998320589096095), (2857, 4.998320589096095), (3022, 4.998320589096095), (3028, 4.998320589096095)]


Now, let's evaluate our developed two-stage model in our evaluation framework:

In [12]:
from src.evaluation import EvaluationPipeline

In [294]:
eval_baseline = EvaluationPipeline(df_ratings.sample(frac=0.1, random_state=5), 0.2)

In [295]:
metrics_output_dict_two_stage = eval_baseline.evaluate( # recommendation_results_baseline
    TwoStageSwingBasedModel(),
    user_average_metrics=False)

  0%|          | 0/19957 [00:00<?, ?it/s]

  0%|          | 0/90397 [00:00<?, ?it/s]

  0%|          | 0/793 [00:00<?, ?it/s]

  0%|          | 0/1556 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/52 [00:00<?, ?it/s]

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [296]:
metrics_output_dict_two_stage

{'mae': 3.5194059619980163,
 'rmse': 3.711459027412266,
 'precision': 0.0,
 'average_precision': 0.6230999004197277,
 'mean_reciprocal_rank': 0.0005236844319878925,
 'ndcg': 0.9417192407629446,
 'coverage': 0.13476726600571332}

In [297]:
with open('two_stage_swing_based_metrics.json', 'w') as f:
    json.dump(metrics_output_dict_two_stage, f)

The main takeaways from these results are the following:
o
- The overall selection of the correct items is the defining factor here - as shown by the high average precision. This shows how the candidate generation stage helped properly selecting the items.
- The ratings themselves were still not the main focus of the model, as evidenced by the `MAE` and `RMSE`, as they were primarily aimed at forming the ranking. This highlights, how the ranking step can actually be detrimental to the rating prediction. In order to increase it we can develop a way to calculate it through the other users' ratings directly.
- The evaluation took a really long time. In fact, it was planned to improve this model iteratively several another times, but the long time of computation played a part in this. Therefore, it as the next step in our two-stage model improvement it was decided to adress this specific issue, and also to further improve the candidate generation step.
- Low top-1 precision indicates how the ranking step rarely correctly selects the correct highest ranked recommendation. However, here a huge part plays a fact that our data has a very low discretization, with 5-rated movies essentially being equally highest-ranked. This very reason also contributes to the low mean reciprocal rank despite the high NDCG, as the second one is more approximately-oriented. It also shows how in general, the model has gotten the relation between the items correctly while not positioning them on the respective positions exactly.

Advantages and drawbacks of the method itself, as well the continuation of the metrics overview, will continue in and after the Experiment 3.4, in which we develop another two-stage recommender system.